# `OrderedDict` — Advanced Tutorial Problems With Solutions

This notebook is intentionally written in a **tutorial style**.

Instead of jumping directly to a large finished implementation, each problem is broken into small logical steps:

1. introduce one behavior,
2. run a tiny experiment,
3. explain the result,
4. build a small helper,
5. combine helpers into a larger solution,
6. test edge cases,
7. discuss design choices and complexity.

The main topic is `collections.OrderedDict`.

## Modern Python note

Modern Python `dict` preserves insertion order, so a normal dictionary is usually enough when you only need insertion order.

`OrderedDict` remains especially useful when the algorithm repeatedly needs operations such as:

- moving an existing key to the end,
- moving an existing key to the beginning,
- popping the first item,
- popping the last item,
- expressing order-sensitive equality between two ordered mappings.

We will focus on those behaviors rather than using `OrderedDict` everywhere by default.

In [1]:
from collections import OrderedDict, deque
from dataclasses import dataclass
import random

# Problem 1 — Updating Values vs Updating Positions

A first subtlety is that **changing a value** and **changing a key's position** are separate operations.

Let us start with a small ordered mapping.

In [2]:
d = OrderedDict()
d['alpha'] = 10
d['beta'] = 20
d['gamma'] = 30
d

OrderedDict([('alpha', 10), ('beta', 20), ('gamma', 30)])

The insertion order is:

`alpha -> beta -> gamma`

Now update the value for `beta`.

In [3]:
d['beta'] = 200
d

OrderedDict([('alpha', 10), ('beta', 200), ('gamma', 30)])

The value changed, but the key did **not** move.

That gives us an important rule:

> Assignment changes the mapping value. It does not automatically express a recency change.

If recency should change, we need an explicit movement operation.

In [4]:
d.move_to_end('beta')
d

OrderedDict([('alpha', 10), ('gamma', 30), ('beta', 200)])

Now the order is:

`alpha -> gamma -> beta`

We can turn that distinction into a helper function.

## Step 1 — Basic update helper

We want a function that can either:

- update without repositioning, or
- update and explicitly move the key to the end.

In [5]:
def update_value(mapping, key, value, *, reposition=False):
    mapping[key] = value
    if reposition:
        mapping.move_to_end(key)
    return mapping

## Step 2 — Test both modes

In [6]:
d = OrderedDict([('a', 1), ('b', 2), ('c', 3)])

update_value(d, 'b', 20, reposition=False)
assert list(d) == ['a', 'b', 'c']

update_value(d, 'a', 10, reposition=True)
assert list(d) == ['b', 'c', 'a']

d

OrderedDict([('b', 20), ('c', 3), ('a', 10)])

### Best-practice takeaway

When ordering is part of program state, make ordering changes explicit. This makes later code easier to reason about and test.

# Problem 2 — Move-to-Front Command Palette

Suppose a small command palette should adapt to usage.

Whenever a command is used successfully, move it to the **front**.

Frequently used commands will naturally drift toward the beginning.

The key operation is:

```python
mapping.move_to_end(key, last=False)
```

Despite the method name, `last=False` means "move to the beginning".

In [7]:
commands = OrderedDict([
    ('open', 'Open a file'),
    ('save', 'Save the file'),
    ('close', 'Close the file'),
    ('search', 'Search in the file'),
])

commands.move_to_end('search', last=False)
commands

OrderedDict([('search', 'Search in the file'),
             ('open', 'Open a file'),
             ('save', 'Save the file'),
             ('close', 'Close the file')])

Now `search` is the first key.

Let us build a reusable abstraction one method at a time.

## Step 1 — Store commands and support a non-mutating lookup

In [8]:
class CommandPalette:
    def __init__(self, commands=()):
        self._commands = OrderedDict(commands)

    def peek(self, name):
        return self._commands[name]

`peek()` should not affect ordering.

Now add the method that records actual usage.

## Step 2 — Move successful commands to the front

In [9]:
class CommandPalette:
    def __init__(self, commands=()):
        self._commands = OrderedDict(commands)

    def peek(self, name):
        return self._commands[name]

    def use(self, name):
        value = self._commands[name]
        self._commands.move_to_end(name, last=False)
        return value

    def add(self, name, description):
        self._commands[name] = description

    def order(self):
        return list(self._commands)

## Step 3 — Walk through several accesses

In [10]:
palette = CommandPalette([
    ('open', 'Open a file'),
    ('save', 'Save the file'),
    ('close', 'Close the file'),
    ('search', 'Search in the file'),
])

assert palette.order() == ['open', 'save', 'close', 'search']

palette.use('search')
assert palette.order() == ['search', 'open', 'save', 'close']

palette.use('save')
assert palette.order() == ['save', 'search', 'open', 'close']

before = palette.order()
assert palette.peek('open') == 'Open a file'
assert palette.order() == before

palette.order()

['save', 'search', 'open', 'close']

### Discussion

This is not an eviction cache. Nothing is removed.

`OrderedDict` is simply being used as a dynamic ordered index where a successful action changes a key's position.

# Problem 3 — Bounded Unique Notification Feed

Now consider a notification feed.

Requirements:

- each notification has a unique ID,
- repeated IDs should not create duplicates,
- seeing the same ID again makes it newest,
- only the most recent `N` unique notifications are kept,
- when the limit is exceeded, remove the oldest item.

We need two order operations:

1. move an existing key to the end,
2. pop the first key/value pair.

Let us isolate the second behavior first.

In [11]:
feed = OrderedDict([
    ('n1', 'Server started'),
    ('n2', 'Backup complete'),
    ('n3', 'New login'),
])

removed = feed.popitem(last=False)
print('Removed:', removed)
print('Remaining:', feed)

Removed: ('n1', 'Server started')
Remaining: OrderedDict({'n2': 'Backup complete', 'n3': 'New login'})


`popitem(last=False)` removes the **first** pair.

If the left side means "oldest", this is a natural eviction operation.

## Step 1 — Validate the size limit

In [12]:
class RecentNotifications:
    def __init__(self, limit):
        if not isinstance(limit, int):
            raise TypeError('limit must be an integer')
        if limit <= 0:
            raise ValueError('limit must be greater than zero')

        self.limit = limit
        self._items = OrderedDict()

## Step 2 — Record or refresh a notification

Assignment updates the message. `move_to_end()` then marks the ID as newest.

In [13]:
class RecentNotifications:
    def __init__(self, limit):
        if not isinstance(limit, int):
            raise TypeError('limit must be an integer')
        if limit <= 0:
            raise ValueError('limit must be greater than zero')

        self.limit = limit
        self._items = OrderedDict()

    def record(self, notification_id, message):
        self._items[notification_id] = message
        self._items.move_to_end(notification_id)

## Step 3 — Enforce the capacity

In [14]:
class RecentNotifications:
    def __init__(self, limit):
        if not isinstance(limit, int):
            raise TypeError('limit must be an integer')
        if limit <= 0:
            raise ValueError('limit must be greater than zero')

        self.limit = limit
        self._items = OrderedDict()

    def record(self, notification_id, message):
        self._items[notification_id] = message
        self._items.move_to_end(notification_id)

        if len(self._items) > self.limit:
            return self._items.popitem(last=False)

        return None

    def snapshot(self):
        return list(self._items.items())

## Step 4 — Trace the order manually

In [15]:
feed = RecentNotifications(3)
feed.record('n1', 'A')
feed.record('n2', 'B')
feed.record('n3', 'C')

assert feed.snapshot() == [('n1', 'A'), ('n2', 'B'), ('n3', 'C')]
feed.snapshot()

[('n1', 'A'), ('n2', 'B'), ('n3', 'C')]

Now refresh `n1`.

It should move from oldest to newest.

In [16]:
feed.record('n1', 'A updated')
assert feed.snapshot() == [('n2', 'B'), ('n3', 'C'), ('n1', 'A updated')]
feed.snapshot()

[('n2', 'B'), ('n3', 'C'), ('n1', 'A updated')]

Now insert one more unique ID.

The feed temporarily has four unique IDs, so the oldest (`n2`) must be evicted.

In [17]:
removed = feed.record('n4', 'D')

assert removed == ('n2', 'B')
assert feed.snapshot() == [
    ('n3', 'C'),
    ('n1', 'A updated'),
    ('n4', 'D'),
]

print('Removed:', removed)
print('Remaining:', feed.snapshot())

Removed: ('n2', 'B')
Remaining: [('n3', 'C'), ('n1', 'A updated'), ('n4', 'D')]


### Why this works well

The algorithm does not scan the full mapping to find the oldest item. The ordering itself carries that information.

# Problem 4 — Deduplicate by Last Appearance

Most introductory deduplication keeps the **first** occurrence.

Here we want a different rule:

> Keep one copy of every value, but order values according to their final appearance.

For example:

```text
red, blue, red, green, blue
```

should become:

```text
red, green, blue
```

The trick is simple: every time we see an item, move its key to the end.

In [18]:
seen = OrderedDict()

for item in ['red', 'blue', 'red', 'green', 'blue']:
    seen[item] = None
    seen.move_to_end(item)

list(seen)

['red', 'green', 'blue']

The final key order reflects the final-occurrence order.

Let us package this into a function.

In [19]:
def dedupe_by_last_occurrence(items):
    seen = OrderedDict()

    for item in items:
        seen[item] = None
        seen.move_to_end(item)

    return list(seen)

## Step 2 — Test edge cases

In [20]:
assert dedupe_by_last_occurrence([]) == []
assert dedupe_by_last_occurrence([1]) == [1]
assert dedupe_by_last_occurrence([1, 1, 1]) == [1]
assert dedupe_by_last_occurrence(['a', 'b', 'a', 'c', 'b']) == ['a', 'c', 'b']

print('All tests passed.')

All tests passed.


### Comparison with first-occurrence deduplication

For first-occurrence behavior in modern Python, this is concise:

```python
list(dict.fromkeys(items))
```

The interesting part here is that repeated values actively change position.

# Problem 5 — Frequency Table Ordered by Most Recent Appearance

Let us extend the previous idea.

We want both:

- a frequency count,
- ordering by the latest time each item appeared.

For the stream:

```text
A, B, A, C, B, A
```

we expect final order:

```text
C -> B -> A
```

with counts `1, 2, 3` respectively.

## Step 1 — Count and reposition in one loop

In [21]:
def recent_frequency_table(items):
    counts = OrderedDict()

    for item in items:
        counts[item] = counts.get(item, 0) + 1
        counts.move_to_end(item)

    return counts

In [22]:
table = recent_frequency_table(['A', 'B', 'A', 'C', 'B', 'A'])
table

OrderedDict([('C', 1), ('B', 2), ('A', 3)])

The positions tell us recency; the values tell us frequency.

In [23]:
assert list(table.items()) == [('C', 1), ('B', 2), ('A', 3)]

## Step 2 — Read the oldest and newest items without copying everything

The first key can be obtained with `next(iter(mapping))`.

In [24]:
def first_item(mapping):
    if not mapping:
        raise LookupError('mapping is empty')
    key = next(iter(mapping))
    return key, mapping[key]

The final key can be obtained with `next(reversed(mapping))`.

In [25]:
def last_item(mapping):
    if not mapping:
        raise LookupError('mapping is empty')
    key = next(reversed(mapping))
    return key, mapping[key]

In [26]:
assert first_item(table) == ('C', 1)
assert last_item(table) == ('A', 3)

print('First:', first_item(table))
print('Last :', last_item(table))

First: ('C', 1)
Last : ('A', 3)


# Problem 6 — Order-Sensitive Workflow Comparison

Suppose mapping order represents execution order in a pipeline.

Then these two mappings may contain the same data but mean different workflows.

In [27]:
left = OrderedDict([
    ('extract', 'v1'),
    ('transform', 'v2'),
    ('load', 'v1'),
])

right = OrderedDict([
    ('transform', 'v2'),
    ('extract', 'v1'),
    ('load', 'v1'),
])

print('Ordered comparison:', left == right)
print('Mapping-only comparison:', dict(left) == dict(right))

Ordered comparison: False
Mapping-only comparison: True


The mapping contents match, but the ordered sequences do not.

Now we want something more informative than `True` or `False`.

## Step 1 — Convert items to sequences

In [28]:
def first_order_mismatch(a, b):
    a_items = list(a.items())
    b_items = list(b.items())

## Step 2 — Compare every possible position

The sequences may have different lengths, so we use the larger length and fill a missing position with `None`.

In [29]:
def first_order_mismatch(a, b):
    a_items = list(a.items())
    b_items = list(b.items())

    for index in range(max(len(a_items), len(b_items))):
        left_item = a_items[index] if index < len(a_items) else None
        right_item = b_items[index] if index < len(b_items) else None

        if left_item != right_item:
            return index, left_item, right_item

    return None

## Step 3 — Test equal order, reordered data, and different lengths

In [30]:
a = OrderedDict([('x', 1), ('y', 2)])
b = OrderedDict([('x', 1), ('y', 2)])
c = OrderedDict([('y', 2), ('x', 1)])
d = OrderedDict([('x', 1)])

assert first_order_mismatch(a, b) is None
assert first_order_mismatch(a, c) == (0, ('x', 1), ('y', 2))
assert first_order_mismatch(a, d) == (1, ('y', 2), None)

print('Mismatch a/c:', first_order_mismatch(a, c))
print('Mismatch a/d:', first_order_mismatch(a, d))

Mismatch a/c: (0, ('x', 1), ('y', 2))
Mismatch a/d: (1, ('y', 2), None)


### Best-practice note

If you only need a boolean, comparing `list(a.items())` and `list(b.items())` is straightforward.

A diagnostic function is useful when you need to explain *where* two ordered workflows diverge.

# Problem 7 — Tiny Ordered Patch Language

We will create a small command language for reordering a mapping.

Supported commands:

```text
FRONT key
BACK key
DELETE key
SET key value
```

We will represent commands as tuples so we can focus on the ordering logic.

## Step 1 — Understand each primitive operation

In [31]:
state = OrderedDict([('a', 1), ('b', 2), ('c', 3)])

state.move_to_end('c', last=False)  # FRONT c
print('After FRONT c:', state)

state.move_to_end('c', last=True)   # BACK c
print('After BACK c :', state)

After FRONT c: OrderedDict({'c': 3, 'a': 1, 'b': 2})
After BACK c : OrderedDict({'a': 1, 'b': 2, 'c': 3})


Deleting safely can use `pop(key, None)` when absence should be ignored.

In [32]:
state.pop('missing', None)
state

OrderedDict([('a', 1), ('b', 2), ('c', 3)])

## Step 2 — Build the dispatcher

In [33]:
def apply_ordered_commands(initial, commands):
    state = OrderedDict(initial)

    for command in commands:
        operation = command[0]

        if operation == 'FRONT':
            _, key = command
            state.move_to_end(key, last=False)

        elif operation == 'BACK':
            _, key = command
            state.move_to_end(key, last=True)

        elif operation == 'DELETE':
            _, key = command
            state.pop(key, None)

        elif operation == 'SET':
            _, key, value = command
            state[key] = value

        else:
            raise ValueError(f'unknown operation: {operation!r}')

    return state

Notice that `SET` does **not** move an existing key. That follows ordinary assignment behavior.

## Step 3 — Execute a complete patch

In [34]:
initial = [('alpha', 10), ('beta', 20), ('gamma', 30)]
commands = [
    ('FRONT', 'gamma'),
    ('SET', 'beta', 200),
    ('SET', 'delta', 40),
    ('BACK', 'gamma'),
    ('DELETE', 'alpha'),
]

result = apply_ordered_commands(initial, commands)
result

OrderedDict([('beta', 200), ('delta', 40), ('gamma', 30)])

In [35]:
assert list(result.items()) == [
    ('beta', 200),
    ('delta', 40),
    ('gamma', 30),
]

### Extension

If your policy says every `SET` operation should also make the key newest, add:

```python
state.move_to_end(key)
```

after assignment.

# Problem 8 — Task Board With Explicit Promotion

Suppose a task list has two manual actions:

- promote a task to the front,
- demote a task to the end.

Processing always removes the first task.

This is not numeric priority; it is explicit positional priority.

## Step 1 — Represent tasks in order

In [36]:
tasks = OrderedDict([
    ('T1', {'title': 'compile report'}),
    ('T2', {'title': 'send email'}),
    ('T3', {'title': 'restart service'}),
])

tasks

OrderedDict([('T1', {'title': 'compile report'}),
             ('T2', {'title': 'send email'}),
             ('T3', {'title': 'restart service'})])

## Step 2 — Promote one task

In [37]:
tasks.move_to_end('T3', last=False)
assert list(tasks) == ['T3', 'T1', 'T2']
tasks

OrderedDict([('T3', {'title': 'restart service'}),
             ('T1', {'title': 'compile report'}),
             ('T2', {'title': 'send email'})])

## Step 3 — Build a class around the behavior

In [38]:
class TaskBoard:
    def __init__(self):
        self._tasks = OrderedDict()

    def add(self, task_id, payload):
        if task_id in self._tasks:
            raise KeyError(f'task already exists: {task_id!r}')
        self._tasks[task_id] = payload

    def promote(self, task_id):
        self._tasks.move_to_end(task_id, last=False)

    def demote(self, task_id):
        self._tasks.move_to_end(task_id, last=True)

    def process_next(self):
        if not self._tasks:
            raise IndexError('no tasks available')
        return self._tasks.popitem(last=False)

    def snapshot(self):
        return list(self._tasks.items())

## Step 4 — Run a realistic sequence

In [39]:
board = TaskBoard()
board.add('T1', 'compile report')
board.add('T2', 'send email')
board.add('T3', 'restart service')

board.promote('T3')
assert [k for k, _ in board.snapshot()] == ['T3', 'T1', 'T2']

board.demote('T3')
assert [k for k, _ in board.snapshot()] == ['T1', 'T2', 'T3']

assert board.process_next() == ('T1', 'compile report')
print(board.snapshot())

[('T2', 'send email'), ('T3', 'restart service')]


### Design limitation

If you need automatic ranking by numbers, deadlines, or multiple priority classes, a heap or purpose-built priority queue may be a better abstraction.

# Problem 9 — Unique Work Queue With Fast Membership

A `deque` is excellent for queue operations.

But if we frequently need to ask whether a job ID is already waiting, a keyed mapping is attractive because membership is by key.

First, the familiar deque pattern:

In [40]:
q = deque(['job-1', 'job-2', 'job-3'])
print(q.popleft())
print(q)

job-1
deque(['job-2', 'job-3'])


Now the same conceptual queue using ordered keys:

In [41]:
q = OrderedDict((job_id, None) for job_id in ['job-1', 'job-2', 'job-3'])
print(q.popitem(last=False))
print(q)

('job-1', None)
OrderedDict({'job-2': None, 'job-3': None})


## Step 1 — Reject duplicate IDs

In [42]:
class UniqueWorkQueue:
    def __init__(self):
        self._jobs = OrderedDict()

    def enqueue(self, job_id, payload):
        if job_id in self._jobs:
            return False
        self._jobs[job_id] = payload
        return True

## Step 2 — Add dequeue, membership, and reprioritization

In [43]:
class UniqueWorkQueue:
    def __init__(self):
        self._jobs = OrderedDict()

    def enqueue(self, job_id, payload):
        if job_id in self._jobs:
            return False
        self._jobs[job_id] = payload
        return True

    def dequeue(self):
        if not self._jobs:
            raise IndexError('queue is empty')
        return self._jobs.popitem(last=False)

    def reprioritize(self, job_id):
        self._jobs.move_to_end(job_id, last=False)

    def __contains__(self, job_id):
        return job_id in self._jobs

    def snapshot(self):
        return list(self._jobs.items())

## Step 3 — Verify semantics

In [44]:
queue = UniqueWorkQueue()

assert queue.enqueue('J1', 'one') is True
assert queue.enqueue('J2', 'two') is True
assert queue.enqueue('J1', 'duplicate') is False
assert 'J2' in queue

queue.enqueue('J3', 'three')
queue.reprioritize('J3')

assert queue.dequeue() == ('J3', 'three')
assert queue.snapshot() == [('J1', 'one'), ('J2', 'two')]

queue.snapshot()

[('J1', 'one'), ('J2', 'two')]

### Structure choice

Use `deque` for a pure queue.

Use an ordered mapping when unique keyed membership and positional operations are central to the problem.

# Problem 10 — Ordered Merge With Conflict Policies

Suppose several configuration fragments are merged.

When a repeated key appears, there are multiple reasonable policies:

1. overwrite the value but keep the original position,
2. overwrite the value and move the key to the end,
3. reject duplicates.

## Step 1 — Observe normal overwrite behavior

In [45]:
merged = OrderedDict([('host', 'localhost'), ('port', 8000)])
merged['host'] = 'example.com'
merged

OrderedDict([('host', 'example.com'), ('port', 8000)])

The key keeps its position.

To explicitly represent "latest definition wins in both value and position", move the key after assignment.

In [46]:
merged['host'] = 'example.org'
merged.move_to_end('host')
merged

OrderedDict([('port', 8000), ('host', 'example.org')])

## Step 2 — Implement all three policies

In [47]:
def ordered_merge(*mappings, policy='keep_position'):
    allowed = {'keep_position', 'move_to_end', 'error'}
    if policy not in allowed:
        raise ValueError(f'policy must be one of {sorted(allowed)}')

    result = OrderedDict()

    for mapping in mappings:
        for key, value in mapping.items():
            repeated = key in result

            if repeated and policy == 'error':
                raise KeyError(f'duplicate key: {key!r}')

            result[key] = value

            if repeated and policy == 'move_to_end':
                result.move_to_end(key)

    return result

## Step 3 — Compare policy outputs

In [48]:
m1 = OrderedDict([('a', 1), ('b', 2)])
m2 = OrderedDict([('c', 3), ('a', 10)])

keep = ordered_merge(m1, m2, policy='keep_position')
move = ordered_merge(m1, m2, policy='move_to_end')

print('keep_position:', keep)
print('move_to_end :', move)

assert list(keep.items()) == [('a', 10), ('b', 2), ('c', 3)]
assert list(move.items()) == [('b', 2), ('c', 3), ('a', 10)]

keep_position: OrderedDict({'a': 10, 'b': 2, 'c': 3})
move_to_end : OrderedDict({'b': 2, 'c': 3, 'a': 10})


## Step 4 — Confirm the strict duplicate policy

In [49]:
try:
    ordered_merge(m1, m2, policy='error')
except KeyError as exc:
    print('Expected error:', exc)

Expected error: "duplicate key: 'a'"


# Problem 11 — Reverse Processing Without Copying

If insertion order represents chronology, then:

- first key = oldest,
- last key = newest.

Sometimes we want to inspect newest entries first without modifying the mapping.

In [50]:
history = OrderedDict([
    ('v1', 'created'),
    ('v2', 'edited'),
    ('v3', 'approved'),
    ('v4', 'published'),
])

for key in reversed(history):
    print(key, '->', history[key])

v4 -> published
v3 -> approved
v2 -> edited
v1 -> created


No reversed copy of the whole mapping was required.

Now let us stop after the newest `n` entries.

## Step 1 — Build a generator

In [51]:
def newest_items(mapping, n):
    if n < 0:
        raise ValueError('n must be >= 0')

    count = 0
    for key in reversed(mapping):
        if count >= n:
            break
        yield key, mapping[key]
        count += 1

## Step 2 — Test `n = 0`, a partial request, and an oversized request

In [52]:
assert list(newest_items(history, 0)) == []

assert list(newest_items(history, 2)) == [
    ('v4', 'published'),
    ('v3', 'approved'),
]

assert list(newest_items(history, 99)) == [
    ('v4', 'published'),
    ('v3', 'approved'),
    ('v2', 'edited'),
    ('v1', 'created'),
]

print(list(newest_items(history, 3)))

[('v4', 'published'), ('v3', 'approved'), ('v2', 'edited')]


# Problem 12 — Ordered Sliding Window of Distinct Values

We now process a stream and keep only the most recent `k` **distinct** values.

When a value reappears, it becomes newest.

For `k = 3` and stream:

```text
A, B, C, A, D
```

the states should be:

```text
[A]
[A, B]
[A, B, C]
[B, C, A]
[C, A, D]
```

## Step 1 — Update the current item's position

In [53]:
window = OrderedDict()
item = 'A'
window[item] = None
window.move_to_end(item)
list(window)

['A']

## Step 2 — Remove the oldest distinct value when the window grows too large

In [54]:
window = OrderedDict((x, None) for x in ['A', 'B', 'C', 'D'])
window.popitem(last=False)
list(window)

['B', 'C', 'D']

## Step 3 — Combine both ideas into a generator

In [55]:
def sliding_distinct_window(stream, k):
    if k <= 0:
        raise ValueError('k must be > 0')

    window = OrderedDict()

    for item in stream:
        window[item] = None
        window.move_to_end(item)

        if len(window) > k:
            window.popitem(last=False)

        yield list(window)

## Step 4 — Verify every intermediate state

In [56]:
states = list(sliding_distinct_window(['A', 'B', 'C', 'A', 'D'], 3))

expected = [
    ['A'],
    ['A', 'B'],
    ['A', 'B', 'C'],
    ['B', 'C', 'A'],
    ['C', 'A', 'D'],
]

assert states == expected

for state in states:
    print(state)

['A']
['A', 'B']
['A', 'B', 'C']
['B', 'C', 'A']
['C', 'A', 'D']


### What the order means

This is not simply "the last three stream elements".

It is "the last three **distinct values**, ordered by their latest appearance".

# Problem 13 — Order-Aware Change Report

Suppose we have an old ordered configuration and a new one.

We want to report four categories:

- added keys,
- removed keys,
- changed values,
- moved keys.

A key counts as moved if it exists in both mappings but appears at a different index.

## Step 1 — Prepare old and new key sequences

In [57]:
old = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
new = OrderedDict([('b', 20), ('a', 1), ('d', 4)])

old_keys = list(old)
new_keys = list(new)

print(old_keys)
print(new_keys)

['a', 'b', 'c']
['b', 'a', 'd']


## Step 2 — Added and removed keys

In [58]:
old_set = set(old_keys)
new_set = set(new_keys)

added = [key for key in new_keys if key not in old_set]
removed = [key for key in old_keys if key not in new_set]

print('added  :', added)
print('removed:', removed)

added  : ['d']
removed: ['c']


The sets are only used for fast membership checks. We still iterate the ordered key lists so the report preserves meaningful order.

## Step 3 — Changed values

In [59]:
changed = [
    key
    for key in old_keys
    if key in new_set and old[key] != new[key]
]

changed

['b']

## Step 4 — Moved keys

In [60]:
old_pos = {key: i for i, key in enumerate(old_keys)}
new_pos = {key: i for i, key in enumerate(new_keys)}

moved = [
    key
    for key in new_keys
    if key in old_set and old_pos[key] != new_pos[key]
]

moved

['b', 'a']

## Step 5 — Package the complete report

In [61]:
def ordered_change_report(old, new):
    old_keys = list(old)
    new_keys = list(new)

    old_set = set(old_keys)
    new_set = set(new_keys)

    added = [key for key in new_keys if key not in old_set]
    removed = [key for key in old_keys if key not in new_set]

    changed = [
        key
        for key in old_keys
        if key in new_set and old[key] != new[key]
    ]

    old_pos = {key: i for i, key in enumerate(old_keys)}
    new_pos = {key: i for i, key in enumerate(new_keys)}

    moved = [
        key
        for key in new_keys
        if key in old_set and old_pos[key] != new_pos[key]
    ]

    return {
        'added': added,
        'removed': removed,
        'changed': changed,
        'moved': moved,
    }

In [62]:
report = ordered_change_report(old, new)

assert report == {
    'added': ['d'],
    'removed': ['c'],
    'changed': ['b'],
    'moved': ['b', 'a'],
}

report

{'added': ['d'], 'removed': ['c'], 'changed': ['b'], 'moved': ['b', 'a']}

### Best-practice observation

A `set` is perfectly useful for membership bookkeeping inside an order-sensitive algorithm, as long as you do not accidentally use the set itself as the source of output order.

# Problem 14 — A Manual Recency Index With "Touch" Operations

Imagine a registry where reading an item should **not** change order, but an explicit `touch(key)` operation should mark the item as newest.

This is useful when the program wants recency to change only at deliberate synchronization points.

## Step 1 — Create the basic mapping

In [63]:
registry = OrderedDict([
    ('client-A', {'status': 'idle'}),
    ('client-B', {'status': 'idle'}),
    ('client-C', {'status': 'idle'}),
])

registry

OrderedDict([('client-A', {'status': 'idle'}),
             ('client-B', {'status': 'idle'}),
             ('client-C', {'status': 'idle'})])

Reading a value with ordinary lookup does not alter position.

In [64]:
before = list(registry)
_ = registry['client-A']
after = list(registry)

assert before == after
print(after)

['client-A', 'client-B', 'client-C']


## Step 2 — Define an explicit touch operation

In [65]:
def touch(mapping, key):
    mapping.move_to_end(key)

In [66]:
touch(registry, 'client-A')
assert list(registry) == ['client-B', 'client-C', 'client-A']
registry

OrderedDict([('client-B', {'status': 'idle'}),
             ('client-C', {'status': 'idle'}),
             ('client-A', {'status': 'idle'})])

## Step 3 — Build a small class with stale-item inspection

In [67]:
class RecencyRegistry:
    def __init__(self):
        self._data = OrderedDict()

    def set(self, key, value):
        self._data[key] = value

    def get(self, key, default=None):
        return self._data.get(key, default)

    def touch(self, key):
        self._data.move_to_end(key)

    def stalest(self):
        if not self._data:
            raise LookupError('registry is empty')
        key = next(iter(self._data))
        return key, self._data[key]

    def newest(self):
        if not self._data:
            raise LookupError('registry is empty')
        key = next(reversed(self._data))
        return key, self._data[key]

    def snapshot(self):
        return list(self._data.items())

## Step 4 — Confirm that only `touch()` changes recency

In [68]:
r = RecencyRegistry()
r.set('A', 1)
r.set('B', 2)
r.set('C', 3)

assert r.get('A') == 1
assert [k for k, _ in r.snapshot()] == ['A', 'B', 'C']

r.touch('A')
assert [k for k, _ in r.snapshot()] == ['B', 'C', 'A']
assert r.stalest() == ('B', 2)
assert r.newest() == ('A', 1)

r.snapshot()

[('B', 2), ('C', 3), ('A', 1)]

# Problem 15 — Randomized Invariant Testing

Stateful ordered algorithms can appear correct for a few hand-written examples while still hiding subtle bugs.

A useful technique is to compare the implementation against a simple reference model using many random operations.

We will test the `RecentNotifications` structure from earlier.

The reference model will deliberately use a plain list because simplicity matters more than performance in the test oracle.

## Step 1 — Reference-model update

In [69]:
def reference_record(model, limit, notification_id, message):
    # model is a list of [id, message] from oldest to newest
    for i, (existing_id, _) in enumerate(model):
        if existing_id == notification_id:
            model.pop(i)
            break

    model.append([notification_id, message])

    if len(model) > limit:
        return tuple(model.pop(0))

    return None

## Step 2 — Run thousands of randomized updates

In [70]:
rng = random.Random(2026)
limit = 5
feed = RecentNotifications(limit)
model = []

for step in range(3000):
    notification_id = f'n{rng.randrange(10)}'
    message = rng.randrange(100000)

    actual_removed = feed.record(notification_id, message)
    expected_removed = reference_record(model, limit, notification_id, message)

    assert actual_removed == expected_removed
    assert feed.snapshot() == [tuple(item) for item in model]
    assert len(feed.snapshot()) <= limit

print('3,000 randomized operations passed.')

3,000 randomized operations passed.


### Why this test is valuable

The reference model is slower, but extremely easy to understand.

That makes it useful for checking a more efficient stateful implementation across many operation sequences.

# Comparative Exercise — `OrderedDict` vs `deque`

The same data structure should not be used for every ordered problem.

A useful question is:

> What operation is central to the workload?

Choose the better default structure for each case:

1. Pure FIFO queue with frequent left pops.
2. Unique keyed queue with frequent membership checks.
3. Need to move an existing key to the front or back repeatedly.
4. Need only insertion-ordered key/value storage in modern Python.
5. Need a numerical priority queue.

## Solutions

1. **Pure FIFO queue:** usually `deque`.
2. **Unique keyed queue + membership:** an ordered mapping can be a strong fit.
3. **Repeated front/back repositioning by key:** `OrderedDict` is convenient.
4. **Insertion order only:** usually plain `dict`.
5. **Numerical priority:** usually `heapq` or a dedicated priority queue.

The best data structure is the one whose semantics match the problem most directly.

# Mini Drills

Try to answer these without looking back.

1. Move key `x` to the front.
2. Move key `x` to the end.
3. Remove the first pair.
4. Remove the last pair.
5. Read the first key without mutation.
6. Read the last key without mutation.
7. Update a value without moving the key.
8. Update a value and move the key to the end.
9. Reverse-iterate keys.
10. Safely delete a key that might not exist.
11. Build last-occurrence deduplication.
12. Compare ordered mappings by item sequence.

## Mini Drill Solutions

In [71]:
d = OrderedDict([('a', 1), ('x', 2), ('z', 3)])

# 1. Move x to the front.
d.move_to_end('x', last=False)

# 2. Move x to the end.
d.move_to_end('x', last=True)

# 3. Remove the first pair.
copy1 = d.copy()
first_pair = copy1.popitem(last=False)

# 4. Remove the last pair.
copy2 = d.copy()
last_pair = copy2.popitem(last=True)

# 5. Read the first key without mutation.
first_key = next(iter(d))

# 6. Read the last key without mutation.
last_key = next(reversed(d))

# 7. Update a value without moving the key.
d['x'] = 200

# 8. Update a value and move the key to the end.
d['x'] = 201
d.move_to_end('x')

# 9. Reverse-iterate keys.
reverse_keys = list(reversed(d))

# 10. Safely delete a missing key.
d.pop('missing', None)

# 11. Last-occurrence deduplication.
def dedupe_last(items):
    result = OrderedDict()
    for item in items:
        result[item] = None
        result.move_to_end(item)
    return list(result)

# 12. Ordered item-sequence comparison.
a = OrderedDict([('p', 1), ('q', 2)])
b = OrderedDict([('q', 2), ('p', 1)])
same_sequence = list(a.items()) == list(b.items())

print('first_pair:', first_pair)
print('last_pair :', last_pair)
print('first_key :', first_key)
print('last_key  :', last_key)
print('reverse   :', reverse_keys)
print('same seq? :', same_sequence)

first_pair: ('a', 1)
last_pair : ('x', 2)
first_key : a
last_key  : x
reverse   : ['x', 'z', 'a']
same seq? : False


# Final Summary

The most important ideas from these problems are not individual method names. They are the design distinctions behind them.

## 1. Value changes and order changes are different operations

Assignment updates a value. Repositioning should be deliberate.

## 2. Decide what the two ends mean

For example:

- left = oldest, right = newest,
- left = next task, right = deferred task,
- left = highest manual priority, right = lowest manual priority.

Once that meaning is clear, `move_to_end` and `popitem` become much easier to reason about.

## 3. Test order as state

Do not only test that the right keys and values exist. If order is semantically important, assert the exact key sequence as well.

## 4. Do not force every ordered problem into `OrderedDict`

Use `deque`, `dict`, `set`, `heapq`, or another structure when it expresses the required operations more directly.

## 5. Prefer small explicit helpers

Functions such as `touch`, `first_item`, `last_item`, or a clear merge policy make order semantics visible to future readers.

# Further Practice Ideas

1. Add timestamps to `RecencyRegistry` and report both positional recency and wall-clock age.
2. Add a `move_before(key, anchor)` operation by rebuilding only when necessary and discuss the cost.
3. Extend the patch language with `SWAP a b`.
4. Add an eviction callback to `RecentNotifications`.
5. Build a two-ended manual scheduler with explicit promote/demote commands.
6. Compare memory and performance tradeoffs between `dict`, `OrderedDict`, and `deque` for one realistic workload.
7. Write property-based tests for `ordered_merge` policies.
8. Add serialization and restoration while preserving exact order.